# 02 - Enhancement Technique Testing

This notebook evaluates each image enhancement technique on GI endoscopy images. We will:

1. **Test individual techniques** — CLAHE, bilateral denoising, unsharp mask sharpening
2. **Visual comparison** — side-by-side original vs. enhanced
3. **Quality metrics** — measure BRISQUE, noise level, blur, and contrast before/after
4. **Interactive parameter tuning** — explore how parameters affect output quality

The goal is to understand each technique's effect *before* feeding images into the classification pipeline, so we can make informed decisions about the adaptive enhancement parameters.

In [ ]:
import sys
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, str(Path.cwd().parent))

from src.enhancement import apply_clahe, adaptive_clahe
from src.enhancement.denoise import bilateral_filter, adaptive_denoise
from src.enhancement.sharpen import unsharp_mask, adaptive_sharpen
from src.enhancement.pipeline import ImageEnhancer
from src.quality.assessment import detect_blur, estimate_noise, measure_contrast
from src.quality.degradation import add_gaussian_noise, add_gaussian_blur, reduce_contrast

sns.set_theme(style="whitegrid", font_scale=1.1)

# ---------- Configuration ----------
DATA_DIR = Path("../data/raw")
IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg", ".bmp", ".tif"}

# Load a few sample images for testing
sample_images = []
for cls_dir in sorted(DATA_DIR.iterdir()):
    if not cls_dir.is_dir():
        continue
    imgs = sorted([p for p in cls_dir.iterdir() if p.suffix.lower() in IMAGE_EXTENSIONS])
    if imgs:
        sample_images.append(imgs[0])
    if len(sample_images) >= 4:
        break

print(f"Loaded {len(sample_images)} sample images for testing")

In [ ]:
def show_comparison(original, results, titles, suptitle=""):
    """Display original alongside multiple processed versions."""
    n = len(results) + 1
    fig, axes = plt.subplots(1, n, figsize=(5 * n, 5))
    
    axes[0].imshow(cv2.cvtColor(original, cv2.COLOR_BGR2RGB))
    axes[0].set_title("Original")
    axes[0].axis("off")
    
    for ax, img, title in zip(axes[1:], results, titles):
        ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        ax.set_title(title)
        ax.axis("off")
    
    if suptitle:
        fig.suptitle(suptitle, fontsize=14, y=1.02)
    fig.tight_layout()
    plt.show()


def quality_table(images, labels):
    """Print a quality metrics comparison table."""
    print(f"{'Label':<20s} {'Noise':>8s} {'Blur(Lap)':>10s} {'RMS Cont':>10s} {'Michelson':>10s}")
    print("-" * 62)
    for img, label in zip(images, labels):
        noise = estimate_noise(img)
        blur = detect_blur(img)
        contrast = measure_contrast(img)
        print(f"{label:<20s} {noise:>8.1f} {blur:>10.1f} {contrast['rms']:>10.2f} {contrast['michelson']:>10.4f}")

## 1. CLAHE — Contrast Limited Adaptive Histogram Equalization

CLAHE improves local contrast by applying histogram equalization on small tiles, with a clip limit to prevent noise amplification. We apply it in the LAB colour space so only luminance is affected, preserving colour fidelity.

**Parameters to explore:**
- `clip_limit`: Higher values allow more contrast enhancement but risk amplifying noise
- `tile_grid_size`: Smaller tiles give finer local adaptation

In [ ]:
# Test CLAHE with varying clip limits on first sample image
img = cv2.imread(str(sample_images[0]))

clip_limits = [1.0, 2.0, 4.0, 8.0]
results = [apply_clahe(img, clip_limit=cl) for cl in clip_limits]
titles = [f"clip={cl}" for cl in clip_limits]

show_comparison(img, results, titles, suptitle="CLAHE: Effect of Clip Limit")

# Quality metrics
quality_table(
    [img] + results,
    ["Original"] + titles,
)

In [ ]:
# Test CLAHE with varying tile grid sizes
grid_sizes = [(4, 4), (8, 8), (16, 16), (32, 32)]
results = [apply_clahe(img, clip_limit=2.0, tile_grid_size=gs) for gs in grid_sizes]
titles = [f"grid={gs[0]}x{gs[1]}" for gs in grid_sizes]

show_comparison(img, results, titles, suptitle="CLAHE: Effect of Tile Grid Size")

## 2. Denoising — Bilateral Filter and Non-Local Means

Endoscopy images often contain sensor noise, especially in low-light regions. Bilateral filtering preserves edges while smoothing flat regions. For heavy noise, Non-Local Means is more effective but slower.

**Parameters to explore:**
- `sigma_color` / `sigma_space`: Control how aggressively the bilateral filter smooths
- `noise_level`: The adaptive function selects the appropriate strategy

In [ ]:
# Create a noisy version of our test image, then denoise
img = cv2.imread(str(sample_images[0]))
noisy = add_gaussian_noise(img, sigma=30)

# Compare bilateral filter parameters
sigmas = [25, 50, 75, 100]
results = [bilateral_filter(noisy, d=9, sigma_color=s, sigma_space=s) for s in sigmas]
titles = [f"sigma={s}" for s in sigmas]

show_comparison(noisy, results, titles, suptitle="Bilateral Filter: Effect of Sigma (on noisy image)")

quality_table(
    [img, noisy] + results,
    ["Clean Original", "Noisy (sigma=30)"] + titles,
)

In [ ]:
# Test adaptive denoising at different noise levels
noise_levels = [10, 25, 50]
fig, axes = plt.subplots(len(noise_levels), 4, figsize=(20, 5 * len(noise_levels)))

for row, sigma in enumerate(noise_levels):
    noisy = add_gaussian_noise(img, sigma=sigma)
    denoised = adaptive_denoise(noisy, noise_level=sigma)

    for col, (im, title) in enumerate(zip(
        [img, noisy, denoised, cv2.absdiff(noisy, denoised)],
        ["Original", f"Noisy (s={sigma})", "Adaptive Denoise", "Removed Noise"]
    )):
        display = cv2.cvtColor(im, cv2.COLOR_BGR2RGB) if col < 3 else cv2.cvtColor(im, cv2.COLOR_BGR2RGB)
        axes[row][col].imshow(display)
        axes[row][col].set_title(title)
        axes[row][col].axis("off")

fig.suptitle("Adaptive Denoising at Different Noise Levels", fontsize=14, y=1.01)
fig.tight_layout()
plt.show()

## 3. Sharpening — Unsharp Mask

Endoscopic images can suffer from motion blur or optical limitations. Unsharp masking enhances edges by subtracting a blurred copy and adding the difference back. Too much sharpening introduces ringing artifacts at edges.

**Parameters to explore:**
- `amount`: Strength of enhancement (>1.0 for aggressive sharpening)
- `kernel_size` and `sigma`: Control what scale of detail gets enhanced

In [ ]:
# Create a blurry version, then sharpen with varying amounts
img = cv2.imread(str(sample_images[0]))
blurry = add_gaussian_blur(img, kernel_size=7)

amounts = [0.5, 1.0, 1.5, 2.0]
results = [unsharp_mask(blurry, kernel_size=5, sigma=1.0, amount=a) for a in amounts]
titles = [f"amount={a}" for a in amounts]

show_comparison(blurry, results, titles, suptitle="Unsharp Mask: Effect of Amount (on blurred image)")

quality_table(
    [img, blurry] + results,
    ["Clean Original", "Blurred (k=7)"] + titles,
)

In [ ]:
# Adaptive sharpening at different blur levels
blur_scores = [0.1, 0.3, 0.5, 0.8]
fig, axes = plt.subplots(len(blur_scores), 3, figsize=(15, 5 * len(blur_scores)))

for row, bs in enumerate(blur_scores):
    # Simulate different blur severities
    k = max(1, int(bs * 11)) | 1  # odd kernel
    blurred = add_gaussian_blur(img, kernel_size=k)
    sharpened = adaptive_sharpen(blurred, blur_score=bs)

    for col, (im, title) in enumerate(zip(
        [img, blurred, sharpened],
        ["Original", f"Blurred (k={k}, score={bs})", "Adaptive Sharpen"]
    )):
        axes[row][col].imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB))
        axes[row][col].set_title(title)
        axes[row][col].axis("off")

fig.suptitle("Adaptive Sharpening at Different Blur Levels", fontsize=14, y=1.01)
fig.tight_layout()
plt.show()

## 4. Full Adaptive Pipeline

The `ImageEnhancer` class combines all three techniques. It first assesses image quality (noise, contrast, blur), then applies each enhancement stage only as strongly as needed. Let's test the full pipeline on degraded images at different severity levels.

In [ ]:
enhancer = ImageEnhancer()

# Test on multiple sample images with medium degradation
fig, axes = plt.subplots(min(4, len(sample_images)), 4, figsize=(20, 5 * min(4, len(sample_images))))
if len(sample_images) == 1:
    axes = [axes]

for row, img_path in enumerate(sample_images[:4]):
    img = cv2.imread(str(img_path))

    # Apply medium degradation: noise + blur + contrast reduction
    degraded = add_gaussian_noise(img, sigma=25)
    degraded = add_gaussian_blur(degraded, kernel_size=5)
    degraded = reduce_contrast(degraded, gamma=0.6)

    # Assess and enhance
    quality = enhancer.assess_quality(degraded)
    enhanced = enhancer.enhance(degraded, quality=quality)

    for col, (im, title) in enumerate(zip(
        [img, degraded, enhanced, cv2.absdiff(img, enhanced)],
        ["Original", "Degraded", "Enhanced", "Residual (|Orig-Enh|)"]
    )):
        axes[row][col].imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB))
        axes[row][col].set_title(title)
        axes[row][col].axis("off")

fig.suptitle("Full Adaptive Pipeline: Original vs Degraded vs Enhanced", fontsize=14, y=1.01)
fig.tight_layout()
plt.show()

## 5. Quality Metrics Comparison

Let's quantitatively compare quality scores across original, degraded (low/medium/high), and enhanced images to understand the recovery capacity of the pipeline.

In [ ]:
# Collect quality metrics across degradation levels
from src.quality.degradation import DEGRADATION_PRESETS

img = cv2.imread(str(sample_images[0]))
enhancer = ImageEnhancer()

conditions = ["Original"]
noise_scores, blur_scores, contrast_scores = [], [], []

# Original
noise_scores.append(estimate_noise(img))
blur_scores.append(detect_blur(img))
contrast_scores.append(measure_contrast(img)["rms"])

for level in ["low", "medium", "high"]:
    preset = DEGRADATION_PRESETS[level]

    # Degrade
    degraded = add_gaussian_noise(img, sigma=float(preset["noise_sigma"]))
    degraded = add_gaussian_blur(degraded, kernel_size=int(preset["blur_kernel"]))
    degraded = reduce_contrast(degraded, gamma=float(preset["gamma"]))

    conditions.append(f"Degraded ({level})")
    noise_scores.append(estimate_noise(degraded))
    blur_scores.append(detect_blur(degraded))
    contrast_scores.append(measure_contrast(degraded)["rms"])

    # Enhance
    enhanced = enhancer.enhance(degraded)

    conditions.append(f"Enhanced ({level})")
    noise_scores.append(estimate_noise(enhanced))
    blur_scores.append(detect_blur(enhanced))
    contrast_scores.append(measure_contrast(enhanced)["rms"])

# Plot
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
x = range(len(conditions))
colors = ["#2ecc71"] + ["#e74c3c", "#27ae60"] * 3  # red=degraded, green=enhanced

axes[0].bar(x, noise_scores, color=colors, edgecolor="black", linewidth=0.5)
axes[0].set_xticks(x)
axes[0].set_xticklabels(conditions, rotation=45, ha="right")
axes[0].set_ylabel("Noise Level")
axes[0].set_title("Noise (lower is better)")

axes[1].bar(x, blur_scores, color=colors, edgecolor="black", linewidth=0.5)
axes[1].set_xticks(x)
axes[1].set_xticklabels(conditions, rotation=45, ha="right")
axes[1].set_ylabel("Laplacian Variance")
axes[1].set_title("Sharpness (higher is better)")

axes[2].bar(x, contrast_scores, color=colors, edgecolor="black", linewidth=0.5)
axes[2].set_xticks(x)
axes[2].set_xticklabels(conditions, rotation=45, ha="right")
axes[2].set_ylabel("RMS Contrast")
axes[2].set_title("Contrast (higher is better)")

fig.suptitle("Quality Metrics: Original vs Degraded vs Enhanced", fontsize=14, y=1.02)
fig.tight_layout()
plt.show()

## 6. Interactive Parameter Tuning

Use this section to experiment with custom parameter combinations. Modify the values below and re-run to see the effect on a specific image.

In [ ]:
# ============================================================
# MODIFY THESE PARAMETERS AND RE-RUN THIS CELL
# ============================================================

IMAGE_INDEX = 0                      # Which sample image to use (0-3)
NOISE_SIGMA = 25                     # Degradation noise level

# CLAHE parameters
CLAHE_CLIP_LIMIT = 3.0
CLAHE_GRID_SIZE = (8, 8)

# Denoising parameters
DENOISE_D = 9
DENOISE_SIGMA_COLOR = 75
DENOISE_SIGMA_SPACE = 75

# Sharpening parameters
SHARPEN_KERNEL = 5
SHARPEN_SIGMA = 1.0
SHARPEN_AMOUNT = 1.0

# ============================================================

img = cv2.imread(str(sample_images[IMAGE_INDEX]))
degraded = add_gaussian_noise(img, sigma=NOISE_SIGMA)
degraded = add_gaussian_blur(degraded, kernel_size=5)

# Apply each step manually
step1_denoise = bilateral_filter(degraded, d=DENOISE_D,
                                  sigma_color=DENOISE_SIGMA_COLOR,
                                  sigma_space=DENOISE_SIGMA_SPACE)
step2_clahe = apply_clahe(step1_denoise,
                           clip_limit=CLAHE_CLIP_LIMIT,
                           tile_grid_size=CLAHE_GRID_SIZE)
step3_sharpen = unsharp_mask(step2_clahe,
                              kernel_size=SHARPEN_KERNEL,
                              sigma=SHARPEN_SIGMA,
                              amount=SHARPEN_AMOUNT)

# Display pipeline stages
show_comparison(
    degraded,
    [step1_denoise, step2_clahe, step3_sharpen],
    ["After Denoise", "After CLAHE", "After Sharpen"],
    suptitle="Custom Pipeline Stages",
)

# Quality comparison
print("\nQuality metrics at each stage:\n")
quality_table(
    [img, degraded, step1_denoise, step2_clahe, step3_sharpen],
    ["Original", "Degraded", "Denoised", "+CLAHE", "+Sharpen"],
)

## Summary

Key takeaways from enhancement testing:

- **CLAHE**: `clip_limit=2.0-3.0` with `grid=(8,8)` provides a good balance between contrast improvement and noise control for endoscopy images. Higher clip limits (>4.0) start amplifying sensor noise.
- **Denoising**: Bilateral filter works well for low-moderate noise (sigma < 30). For heavy noise, Non-Local Means (triggered automatically by `adaptive_denoise`) preserves texture better.
- **Sharpening**: `amount=1.0-1.5` recovers edge detail without introducing visible ringing. The adaptive function correctly skips near-sharp images.
- **Pipeline order**: Denoise → CLAHE → Sharpen is the correct sequence. Denoising first prevents CLAHE from amplifying noise, and sharpening last recovers edges softened by denoising.
- **Quality recovery**: The adaptive pipeline consistently reduces the quality gap between degraded and original images across all three metrics (noise, sharpness, contrast).

**Next steps**: Run `exp1_baseline.py` and `exp2_enhancement.py` to measure whether these quality improvements translate into better classification accuracy.